# Plotting Monthly Correlation by Region
## By Landon Moeller

### Importing Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.stats import pearsonr
from IPython.display import display
import xarray as xr

### Opening Data

In [2]:
nh_ds = xr.open_dataset("../Data/NH_tele_cli_anoms_detrended_1950-2024.nc")
sh_ds = xr.open_dataset("../Data/SH_tele_cli_anoms_detrended_1950-2024.nc")
global_ds = xr.open_dataset("../Data/Global_tele_cli_anoms_detrended_1950-2024.nc")
nh_ds

<xarray.Dataset> Size: 497kB
Dimensions:                                  (year: 75, month: 12)
Coordinates:
  * year                                     (year) int64 600B 1950 ... 2024
  * month                                    (month) int64 96B 1 2 3 ... 11 12
    time                                     (year, month) datetime64[ns] 7kB ...
Data variables: (12/68)
    TNA                                      (year, month) float64 7kB ...
    WPO                                      (year, month) float64 7kB ...
    WP                                       (year, month) float64 7kB ...
    SCA                                      (year, month) float64 7kB ...
    EAWR                                     (year, month) float64 7kB ...
    EA                                       (year, month) float64 7kB ...
    ...                                       ...
    TP_Anomaly_Western_Australia_Detrended   (year, month) float64 7kB ...
    TP_StdAnom_Western_Australia_Detrended   (year, month) float64 7kB ...
    T2M_Anomaly_Northeast_Detrended          (year, month) float64 7kB ...
    T2M_StdAnom_Northeast_Detrended          (year, month) float64 7kB ...
    TP_Anomaly_Northeast_Detrended           (year, month) float64 7kB ...
    TP_StdAnom_Northeast_Detrended           (year, month) float64 7kB ...
Attributes:
    Name:                   NH Teleconnections and Regional Climate Anomalies...
    Teleconnection_source:  Climate Prediction Center
    Anomaly_source:         ECMWF Reanalysis v5 (ERA5)

### Creating a Spaghetti Plot

Running this widget allows the user to create the plot for a spcecified region and variable.

In [3]:
# Some useful lists for plot generation

nh_indices = ['TNA', 'WPO', 'WP', 'SCA', 'EAWR', 'EA', 'ADI', 'AMO',
              'AO', 'EPNP', 'EPO', 'NAO', 'NOI', 'PDO', 'PNA', 'POL']

sh_indices  = ['SAOD', 'TSA', 'SPOD', 'TPI', 'SOI', 'AAO']

global_indices = ['IPO', 'IOD', 'EMI', 'QBO', 'MEI', 'AAM', 'ENSO_34']

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

nh_regions = ['Midwest', 'Southern_Plains', 'Northeast', 'Western_Europe',
              'Eastern_Europe', 'Black_Sea', 'India', 'East_Asia']

sh_regions = ['Western_Australia', 'Eastern_Australia', 'Northern_Brazil', 'Southern_Brazil', 'Argentina']

variable_map = {'Temperature': 'T2M', 'Precipitation': 'TP'}

# Getting indices by hemisphere
def get_indices(ds):

    if 'TNA' in ds.variables:
        return nh_indices

    if 'SAOD' in ds.variables:
        return sh_indices

    return global_indices

# Computing the mean correlation of both local and global teleconnections
def compute_mean_corr(ds_local, ds_global, region, variable):

    var_name = f"{variable}_StdAnom_{region}_Detrended"

    indices = sorted(set(get_indices(ds_local)) | set(get_indices(ds_global)))

    corr = np.full((len(indices), 12), np.nan)

    for m in range(1, 13):

        y = ds_local.sel(month=m)[var_name].values.ravel()

        for i, idx in enumerate(indices):

            if idx in ds_local:
                x = ds_local.sel(month=m)[idx].values.ravel()

            elif idx in ds_global:
                x = ds_global.sel(month=m)[idx].values.ravel()

            else:
                continue

            valid = ~np.isnan(x) & ~np.isnan(y)

            if valid.sum() < 10:
                continue

            corr[i, m-1] = pearsonr(x[valid], y[valid])[0]

    return pd.DataFrame(corr, index=indices, columns=months)

# Plotting function
def plot_monthly_tele_corrs(ds_local, ds_global, region, variable):

    df = compute_mean_corr(ds_local, ds_global, region, variable)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=400)

    np.random.seed(29)
    colors = np.random.rand(len(df), 3)

    x = np.arange(12)

    for idx, color in zip(df.index, colors):

        y = df.loc[idx].values

        ax.plot(x, y, lw=1, alpha=0.9, color=color, label=idx)

        ax.text(-0.45, y[0], idx, fontsize=8, fontweight='bold', color=color, va='center')

    ax.plot(x, df.mean(axis=0), color='black', lw=3, marker='o', mfc='white', label='Mean')

    ax.axhline(0, color='gray', lw=1)
    ylim = np.nanmax(np.abs(df.values)) + 0.05
    ax.set_ylim(-ylim, ylim)
    ax.set_xticks(x)
    ax.set_xticklabels(months)
    ax.set_ylabel("Pearson r", fontweight='bold')
    ax.set_xlabel("Month", fontweight='bold')

    title_var = ("Temperature" if variable == 'T2M' else "Precipitation")
    ax.set_title(f"Correlations for {title_var} ({region.replace('_',' ')})", fontsize=14, fontweight='bold')

    ax.grid(True, axis='y', ls='--', alpha=0.4)
    ax.legend(bbox_to_anchor=(1.006, 1.012), loc='upper left', fontsize=8.5, framealpha=0.9)

    plt.tight_layout()
    # Extra Code by Tyson Stewart: To help generate pre-generated figures for Everstream, this line of code creates a png for each selected combination using the code from above
    #plt.savefig(f"./Spaghetti Plots/{region.replace('_',' ')}/{region.replace('_',' ')}_{title_var}.png")
    plt.show()

# Creating the widget
def interactive_spaghetti_plot():

    region_widget = widgets.Dropdown(
        options=nh_regions + sh_regions,
        value='Argentina',
        description='Region:'
    )

    variable_widget = widgets.Dropdown(
        options=list(variable_map.keys()),
        value='Precipitation',
        description='Variable:'
    )

    button = widgets.Button(
        description='Build Plot',
        button_style='success'
    )

    output = widgets.Output()

    def build_plot(b):

        with output:

            output.clear_output(wait=True)

            region = region_widget.value
            variable = variable_map[variable_widget.value]

            ds_local = (nh_ds if region in nh_regions else sh_ds)

            plot_monthly_tele_corrs(ds_local, global_ds, region, variable)

    button.on_click(build_plot)

    display(widgets.VBox([
        region_widget,
        variable_widget,
        button,
        output
    ]))

interactive_spaghetti_plot()